In [41]:
import pandas as pd
import numpy as np
import zipfile

with zipfile.ZipFile('dialectro.zip') as zip_ref:
    zip_ref.extractall()

#Load data
train_df = pd.read_csv('train.csv')
test_df = pd.read_csv('test.csv')

train_df.head()

,ID,text,label
0,1,Atât e de înfricoșat de gândești că e turbat a...,graiul moldovenesc
1,3,Cei ce n-au dreptate n-ar mai năzui în veci la...,graiul moldovenesc
2,5,Deoarece ați primit bani de la oaspetele dumne...,graiul moldovenesc
3,7,Primiți vă rog oameni buni această mică mulțăm...,graiul moldovenesc
4,9,Fiin'c-o vint la noi vezi binie baș în zâua dă...,graiul bănățean


# Subtasks 1, 2 & 3

In [ ]:
#Subtask 1
import re

answer_sub1 = 0
for df in [train_df, test_df]:
    num_aparitii = df['text'].str.lower().apply(lambda x: len(re.findall(r'\bpâni\b', x)))
    answer_sub1 += num_aparitii.sum()

print(answer_sub1)

13


I am not aware if this happens in every language, but in romanian it's pretty common to find words in another words.

Therefore you can't do a simple `if 'pâni' in word` as it won't give you the real count.

In [43]:
#Subtask 2
import string

sub2_function = lambda text: len([c for c in text if c in string.punctuation])

mold = train_df.loc[train_df['label'] == 'graiul moldovenesc', 'text'].apply(sub2_function).mean()
banat = train_df.loc[train_df['label'] == 'graiul bănățean', 'text'].apply(sub2_function).mean()

answer_sub2 = round(abs(banat-mold),2)

print(answer_sub2)

0.05


In [44]:
#Subtask 3
def count_diacritice(text):
    diacritice = 'ăâîșț'
    num_diacritice = 0
    for c in text.lower():
        if c in diacritice:
            num_diacritice += 1
    return num_diacritice

answer_sub3 = list(test_df['text'].apply(count_diacritice))
print(answer_sub3[:10])

[2, 3, 3, 5, 8, 3, 5, 6, 9, 5]


# Subtask 4

In [45]:
#Preprocess
def clean_text(text):
    text = text.lower()
    #Remove nums
    text = re.sub(r'\d+', '', text)
    #Extra whitespaces
    text = re.sub(r'\s+', ' ', text).strip()
    return text

#Ready to apply new features
X_train = pd.DataFrame(train_df['text'].apply(clean_text), columns=['text'])
X_test = pd.DataFrame(test_df['text'].apply(clean_text), columns=['text'])

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer

vectorizer = TfidfVectorizer(
    analyzer='char_wb',
    max_features=30000, 
    ngram_range=(2, 5),
    sublinear_tf=True,
)

In [ ]:
#Create new features
from scipy.sparse import hstack, csr_matrix
from sklearn.preprocessing import MaxAbsScaler, LabelEncoder

def nlp_feature_eng(df):
    features = pd.DataFrame(index=df.index)
    text_col = df['text'].str.lower()

    features['text_length'] = df['text'].apply(len)
    features['word_count'] = df['text'].str.split().apply(len)
    features['sentence_count'] = df['text'].apply(
        lambda x: max(len(re.findall(r'[.!?]+', x)), 1)
    )

    features['num_punctuation'] = text_col.apply(
        lambda x: sum(1 for c in x if c in string.punctuation)
    )
    diacritice = "ăâîșț"
    features['num_diacritice'] = text_col.apply(
        lambda x: sum(1 for c in x if c in diacritice)
    )
    features['num_cratime'] = text_col.apply(
        lambda x: x.count('-')
    )
    features['num_apostroafe'] = text_col.apply(
        lambda x: x.count("'")
    )
    return features


train_feats = nlp_feature_eng(X_train)
test_feats = nlp_feature_eng(X_test)

scaler = MaxAbsScaler()
X_train_scaled = scaler.fit_transform(train_feats)
X_test_scaled = scaler.transform(test_feats)

X_vectorized = vectorizer.fit_transform(X_train['text']).tocoo()
test_vectorized = vectorizer.transform(X_test['text']).tocoo()

X_final = hstack([X_vectorized, csr_matrix(X_train_scaled)])
test_final = hstack([test_vectorized, csr_matrix(X_test_scaled)])

The additional features(or at least some) are needed to get 100p.

In [48]:
#Model
from sklearn.svm import LinearSVC
from sklearn.metrics import f1_score
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import GridSearchCV

le = LabelEncoder()
y_train= le.fit_transform(train_df['label'])

model = LogisticRegression(class_weight='balanced', random_state=42)

param_grid = {
    'C':[0.01, 0.1, 1, 5, 10],
    'max_iter':[1000, 1500, 2000]
}

search = GridSearchCV(
    model, param_grid=param_grid,
    cv=5, scoring='f1_macro',
    n_jobs=-1, verbose=1
)
search.fit(X_final, y_train)
model = search.best_estimator_

model.fit(X_final, y_train)
preds = model.predict(test_final)

Fitting 5 folds for each of 15 candidates, totalling 75 fits


In [ ]:
#Submission
output_df = pd.DataFrame({
    'subtaskID':[1, 2] + [3] *len(test_df) + [4] * len(test_df),
    'datapointID':[1,1] + list(test_df['ID']) * 2,
    'answer':[answer_sub1, answer_sub2] + answer_sub3 + list(le.inverse_transform(preds))
})

output_df.to_csv('submission.csv', index=False)
output_df.head()

,subtaskID,datapointID,answer
0,1,1,13
1,2,1,0.05
2,3,1491,2
3,3,1493,3
4,3,1495,3


100p/100p; F1: 0.99